In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import nflreadpy as nfl

In [2]:
MIN_GAMES = 20
START_YEAR = 2024
END_YEAR = 2026
DEF_POSITIONS = {'DE', 'DT', 'NT', 'DL', 'OLB', 'ILB', 'MLB', 'LB', 'CB', 'S', 'FS', 'SAF', 'DB'}
OF_POSITIONS = {'RB', 'FB', 'WR', 'TE', 'OT', 'G', 'C'}
ST_POSITIONS = {'K', 'P'}

raw = nfl.load_player_stats(seasons = list(range(START_YEAR, END_YEAR + 1)), summary_level = 'reg').to_pandas()
list(raw.columns)

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'season_type',
 'recent_team',
 'games',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'receiv

Not all players have two word names (ex. Jordan van den Berg). I will make first name the first word and last name the rest of the words. 

In [3]:
player_stats = (
    raw
    .sort_values(by = 'season', ascending = True)
    .groupby('player_id', as_index = False)
    .agg(
        name = ('player_display_name', 'first'),
        position = ('position', 'first'),
        total_teams = ('recent_team', 'nunique'),
        current_team = ('recent_team', 'last'),
        games = ('games', 'sum'),
        completions = ('completions', 'sum'),
        attempts = ('attempts', 'sum'),
        passing_yards = ('passing_yards', 'sum'),
        passing_tds = ('passing_tds', 'sum'),
        passing_interceptions = ('passing_interceptions', 'sum'),
        passing_epa = ('passing_epa', 'sum'),
        carries = ('carries', 'sum'),
        rushing_yards = ('rushing_yards', 'sum'),
        rushing_tds = ('rushing_tds', 'sum'),
        rushing_epa = ('rushing_epa', 'sum'),
        receptions = ('receptions', 'sum'),
        targets = ('targets', 'sum'),
        receiving_yards = ('receiving_yards', 'sum'),
        receiving_tds = ('receiving_tds', 'sum'),
        receiving_epa = ('receiving_epa', 'sum'),
        def_tackles_solo = ('def_tackles_solo', 'sum'),
        def_tackles_with_assist = ('def_tackles_with_assist', 'sum'),
        def_tackle_assists = ('def_tackle_assists', 'sum'),
        def_sacks = ('def_sacks', 'sum'),
        def_sack_yards = ('def_sack_yards', 'sum'),
        def_qb_hits = ('def_qb_hits', 'sum'),
        def_interceptions = ('def_interceptions', 'sum'),
        fg_made = ('fg_made', 'sum'),
        fg_att = ('fg_att', 'sum'),
        fg_long = ('fg_long', 'sum'),
        fg_made_distance = ('fg_made_distance', 'sum')
    )
)

player_stats['first_name'] = [name.split()[0] for name in player_stats['name']]
player_stats['last_name'] = [" ".join(name.split()[1:]) for name in player_stats['name']]

player_stats.head()

,player_id,name,position,total_teams,current_team,games,completions,attempts,passing_yards,passing_tds,...,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,fg_made,fg_att,fg_long,fg_made_distance,first_name,last_name
0,00-0022942,Philip Rivers,QB,1,IND,3,58,92,544,4,...,0.0,0.0,0,0,0,0,0.0,0,Philip,Rivers
1,00-0023459,Aaron Rodgers,QB,2,PIT,42,885,1399,9233,67,...,0.0,0.0,0,0,0,0,0.0,0,Aaron,Rodgers
2,00-0023853,Matt Prater,K,2,BUF,19,0,0,0,0,...,0.0,0.0,0,0,24,26,109.0,923,Matt,Prater
3,00-0024243,Marcedes Lewis,TE,2,DEN,6,0,0,0,0,...,0.0,0.0,0,0,0,0,0.0,0,Marcedes,Lewis
4,00-0025565,Nick Folk,K,3,ATL,39,0,0,0,0,...,0.0,0.0,0,0,62,69,165.0,2459,Nick,Folk


In [4]:
player_stats_qualified = player_stats.loc[player_stats['games'] >= MIN_GAMES]

print(f'We have {player_stats_qualified.shape[0]} NFL players who have played at least {MIN_GAMES} games since {START_YEAR}.')

We have 1056 NFL players who have played at least 20 games since 2024.


In [5]:
player_stats_qualified.to_csv('player_stats_qualified.csv', index = False)
player_stats_qualified[['first_name', 'last_name']].to_csv('player_list.csv', index = False)